# CarbinWatcher — Trash Detector Training

Fine-tunes **YOLOv8x** (extra-large, largest YOLOv8 variant) on the **TACO** (Trash Annotations in Context) open-source dataset and exports to **ONNX** for edge inference.  
No API key required — TACO is freely available under **CC BY 4.0**.  
Run on Google Colab (free T4 GPU) or locally with a CUDA-capable GPU.

> **Dataset:** Proença & Simões, 2020 · [github.com/pedropro/TACO](https://github.com/pedropro/TACO) · ~1 500 images, 60 COCO categories  
> TACO categories are remapped to the 20 canonical classes below.

> **Model choice:** YOLOv8x is the largest YOLOv8 model (~68 MB ONNX, ~1.5–2 GB RAM at runtime).  
> It fits comfortably within the 4 GB RAM envelope of the Arduino UNO Q while maximising detection accuracy.  
> Training compute is not a concern — only inference cost on the edge device.

## Classes → bin mapping

| Category | Classes |
|----------|---------|
| recycle  | plastic_bottle, glass_bottle, metal_can, cardboard, paper, newspaper, aluminum_foil, beverage_carton |
| compost  | food_waste, fruit_peel, coffee_grounds, eggshell |
| landfill | styrofoam, plastic_bag, straw, tissue, chip_bag, dirty_container |
| hazardous | battery, electronics |

In [ ]:
# Install dependencies (Colab / fresh venv)
import subprocess, sys

packages = [
    'ultralytics>=8.0',
    'onnx>=1.14',
    'onnxruntime>=1.17',
    'requests',
    'matplotlib',
    'seaborn',
    'scikit-learn',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import onnxruntime as ort
import yaml
from ultralytics import YOLO

# ── Project paths ──────────────────────────────────────────────────────────
ROOT        = Path('carbinwatcher_data')
MODELS_DIR  = Path('../edge/linux/models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# ── Class definitions ──────────────────────────────────────────────────────
CLASSES = [
    'plastic_bottle',    # 0  recycle
    'glass_bottle',      # 1  recycle
    'metal_can',         # 2  recycle
    'cardboard',         # 3  recycle
    'paper',             # 4  recycle
    'newspaper',         # 5  recycle
    'aluminum_foil',     # 6  recycle
    'beverage_carton',   # 7  recycle
    'food_waste',        # 8  compost
    'fruit_peel',        # 9  compost
    'coffee_grounds',    # 10 compost
    'eggshell',          # 11 compost
    'styrofoam',         # 12 landfill
    'plastic_bag',       # 13 landfill
    'straw',             # 14 landfill
    'tissue',            # 15 landfill
    'chip_bag',          # 16 landfill
    'dirty_container',   # 17 landfill
    'battery',           # 18 hazardous
    'electronics',       # 19 hazardous
]

LABEL_TO_CATEGORY = {
    'plastic_bottle': 'recycle',  'glass_bottle':   'recycle',
    'metal_can':      'recycle',  'cardboard':      'recycle',
    'paper':          'recycle',  'newspaper':      'recycle',
    'aluminum_foil':  'recycle',  'beverage_carton':'recycle',
    'food_waste':     'compost',  'fruit_peel':     'compost',
    'coffee_grounds': 'compost',  'eggshell':       'compost',
    'styrofoam':      'landfill', 'plastic_bag':    'landfill',
    'straw':          'landfill', 'tissue':         'landfill',
    'chip_bag':       'landfill', 'dirty_container':'landfill',
    'battery':        'hazardous','electronics':    'hazardous',
}

print(f'Tracking {len(CLASSES)} classes across 4 bin categories')

## 1. Dataset — TACO (Trash Annotations in Context)

**Open-source, no API key required.**

The cell below:
1. Downloads the TACO annotation manifest (~15 MB) from GitHub.
2. Maps TACO's 60 COCO categories to our 20 canonical classes (unmapped categories are skipped).
3. Downloads up to `MAX_TRAIN_IMGS` / `MAX_VAL_IMGS` images from their source URLs.
4. Converts COCO `[x, y, w, h]` bounding boxes to YOLO normalised format.
5. Writes a `data.yaml` pointing at the prepared splits.

Lower the image limits for a quick smoke-test; set them to `None` to use the full corpus.

In [ ]:
import json
import random
import requests
import yaml

# ── Download limits ────────────────────────────────────────────────────────
# Set to None to use the full ~1 500-image TACO corpus.
MAX_TRAIN_IMGS = 200
MAX_VAL_IMGS   = 50

# ── TACO category name → canonical class ──────────────────────────────────
TACO_TO_CLASS = {
    'Plastic bottle':           'plastic_bottle',
    'Bottle':                   'plastic_bottle',
    'Glass bottle':             'glass_bottle',
    'Broken glass':             'glass_bottle',
    'Metal can':                'metal_can',
    'Drink can':                'metal_can',
    'Food Can':                 'metal_can',
    'Aluminium foil':           'aluminum_foil',
    'Drink carton':             'beverage_carton',
    'Juice carton':             'beverage_carton',
    'Milk carton':              'beverage_carton',
    'Corrugated carton':        'cardboard',
    'Egg carton':               'cardboard',
    'Paper':                    'paper',
    'Book':                     'paper',
    'Newspaper':                'newspaper',
    'Plastic bag + wrapper':    'plastic_bag',
    'Bag':                      'plastic_bag',
    'Garbage bag':              'plastic_bag',
    'Plastic film':             'plastic_bag',
    'Straw':                    'straw',
    'Paper straw':              'straw',
    'Tissues':                  'tissue',
    'Crisp packet':             'chip_bag',
    'Polystyrene item':         'styrofoam',
    'Styrofoam piece':          'styrofoam',
    'Foam food container':      'styrofoam',
    'Plastic container':        'dirty_container',
    'Dry food container':       'dirty_container',
    'Food container':           'dirty_container',
    'Spread tub':               'dirty_container',
    'Battery':                  'battery',
    'Blister pack':             'electronics',
}

CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}

TACO_ANN_URL = (
    'https://raw.githubusercontent.com/pedropro/TACO/master/data/annotations.json'
)

ROOT.mkdir(parents=True, exist_ok=True)
ann_path = ROOT / 'taco_annotations.json'

if not ann_path.exists():
    print('Fetching TACO annotations (~15 MB)…')
    r = requests.get(TACO_ANN_URL, timeout=120)
    r.raise_for_status()
    ann_path.write_bytes(r.content)
    print('Saved →', ann_path)
else:
    print('Annotations already cached →', ann_path)

with open(ann_path) as f:
    taco = json.load(f)

# ── Map TACO category IDs to canonical classes ─────────────────────────────
cat_id_to_class: dict[int, str] = {}
for cat in taco['categories']:
    mapped = TACO_TO_CLASS.get(cat['name'])
    if mapped:
        cat_id_to_class[cat['id']] = mapped

# ── Group annotations by image_id ─────────────────────────────────────────
img_anns: dict[int, list[dict]] = {}
for ann in taco['annotations']:
    if ann['category_id'] in cat_id_to_class:
        img_anns.setdefault(ann['image_id'], []).append(ann)

relevant_imgs = [img for img in taco['images'] if img['id'] in img_anns]
random.seed(42)
random.shuffle(relevant_imgs)

n_train = MAX_TRAIN_IMGS or int(len(relevant_imgs) * 0.8)
n_val   = MAX_VAL_IMGS   or (len(relevant_imgs) - n_train)
split_imgs = {
    'train': relevant_imgs[:n_train],
    'valid': relevant_imgs[n_train : n_train + n_val],
}
print(f'Relevant images — total: {len(relevant_imgs)}  '
      f'train: {len(split_imgs["train"])}  val: {len(split_imgs["valid"])}')
print(f'Mapped categories: {len(cat_id_to_class)} / {len(taco["categories"])}')


def coco_box_to_yolo(bbox: list, img_w: int, img_h: int) -> tuple:
    """COCO [x, y, w, h] → YOLO [cx, cy, bw, bh] normalised."""
    x, y, bw, bh = bbox
    return (x + bw / 2) / img_w, (y + bh / 2) / img_h, bw / img_w, bh / img_h


def download_split(split: str, images: list[dict]) -> None:
    img_dir = ROOT / split / 'images'
    lbl_dir = ROOT / split / 'labels'
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)

    ok = 0
    for img_info in images:
        img_id   = img_info['id']
        url      = img_info.get('flickr_url') or img_info.get('coco_url', '')
        stem     = f'{img_id:06d}'
        img_path = img_dir / f'{stem}.jpg'
        lbl_path = lbl_dir / f'{stem}.txt'

        if not img_path.exists():
            try:
                resp = requests.get(url, timeout=30)
                resp.raise_for_status()
                img_path.write_bytes(resp.content)
            except Exception as e:
                print(f'  skip {img_id}: {e}')
                continue

        w, h  = img_info['width'], img_info['height']
        lines = []
        for ann in img_anns.get(img_id, []):
            cls_name = cat_id_to_class[ann['category_id']]
            cls_idx  = CLASS_TO_IDX[cls_name]
            cx, cy, bw, bh = coco_box_to_yolo(ann['bbox'], w, h)
            lines.append(f'{cls_idx} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')

        lbl_path.write_text('\n'.join(lines) + '\n')
        ok += 1

    print(f'  {split}: {ok} images ready')


print('\nDownloading images (this may take a few minutes)…')
download_split('train', split_imgs['train'])
download_split('valid', split_imgs['valid'])

data_yaml = {
    'path':  str(ROOT.resolve()),
    'train': 'train/images',
    'val':   'valid/images',
    'nc':    len(CLASSES),
    'names': CLASSES,
}
DATASET_YAML = ROOT / 'data.yaml'
with open(DATASET_YAML, 'w') as f:
    yaml.dump(data_yaml, f)
print('\nDataset ready →', DATASET_YAML)

## 2. Verify data.yaml

The download cell already writes `data.yaml` with the canonical class list.
This cell confirms the config looks correct before training.

In [ ]:
with open(DATASET_YAML) as f:
    data_cfg = yaml.safe_load(f)

print('Original classes:', data_cfg.get('names', []))

# If nc matches, assume names are already aligned.
# Otherwise update names (and nc) to our canonical list — verify alignment manually.
if data_cfg.get('nc', 0) != len(CLASSES):
    print(f'⚠️  nc mismatch ({data_cfg["nc"]} vs {len(CLASSES)}) — updating names to canonical list.')
    print('   Verify that label indices in your dataset match CLASSES order above!')
    data_cfg['nc']    = len(CLASSES)
    data_cfg['names'] = CLASSES
    with open(DATASET_YAML, 'w') as f:
        yaml.dump(data_cfg, f)

print('\nFinal classes:', data_cfg['names'])

## 3. Fine-tune YOLOv8x

YOLOv8x is the **extra-large** variant (68.2 M parameters).  
Training on Colab T4 (15 GB VRAM) — use `BATCH_SIZE = 8` to stay within GPU memory.  
Inference on the Arduino UNO Q uses only ~1.5–2 GB of its 4 GB RAM budget.

In [ ]:
# Hyper-parameters
EPOCHS      = 50
IMG_SIZE    = 640
BATCH_SIZE  = 8      # YOLOv8x needs ~14 GB VRAM at batch 16; use 8 for T4 safety
LR0         = 0.01
PATIENCE    = 10     # early-stopping patience (epochs without improvement)
PROJECT_DIR = 'runs/carbinwatcher'
RUN_NAME    = 'yolov8x_trash'

model = YOLO('yolov8x.pt')  # download pretrained extra-large weights

results = model.train(
    data      = str(DATASET_YAML),
    epochs    = EPOCHS,
    imgsz     = IMG_SIZE,
    batch     = BATCH_SIZE,
    lr0       = LR0,
    patience  = PATIENCE,
    project   = PROJECT_DIR,
    name      = RUN_NAME,
    # Augmentation
    hsv_h     = 0.015,
    hsv_s     = 0.7,
    hsv_v     = 0.4,
    flipud    = 0.0,
    fliplr    = 0.5,
    mosaic    = 1.0,
    mixup     = 0.1,
    copy_paste= 0.1,
    degrees   = 10.0,
    translate = 0.1,
    scale     = 0.5,
    shear     = 2.0,
    # Device
    device    = 0 if __import__('torch').cuda.is_available() else 'cpu',
    verbose   = True,
)

BEST_WEIGHTS = Path(PROJECT_DIR) / RUN_NAME / 'weights' / 'best.pt'
print('\nBest weights saved →', BEST_WEIGHTS)

## 4. Evaluate

In [ ]:
best_model = YOLO(str(BEST_WEIGHTS))
metrics    = best_model.val(data=str(DATASET_YAML), imgsz=IMG_SIZE, verbose=True)

print(f'\nmAP@50:     {metrics.box.map50:.4f}')
print(f'mAP@50-95:  {metrics.box.map:.4f}')
print(f'Precision:  {metrics.box.mp:.4f}')
print(f'Recall:     {metrics.box.mr:.4f}')

In [ ]:
# Per-class AP bar chart
ap_per_class = metrics.box.ap50  # shape [num_classes]
cat_colors = {
    'recycle':   '#4CAF50',
    'compost':   '#8D6E63',
    'landfill':  '#9E9E9E',
    'hazardous': '#F44336',
}
colors = [cat_colors[LABEL_TO_CATEGORY[c]] for c in CLASSES]

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(CLASSES, ap_per_class, color=colors)
ax.set_ylabel('AP@50')
ax.set_title('Per-class AP@50')
ax.set_xticklabels(CLASSES, rotation=45, ha='right', fontsize=8)
ax.axhline(float(np.mean(ap_per_class)), color='navy', linestyle='--', label='mean')
ax.legend()
plt.tight_layout()
plt.savefig('ap_per_class.png', dpi=150)
plt.show()

## 5. Export to ONNX

In [ ]:
onnx_path = best_model.export(
    format   = 'onnx',
    imgsz    = IMG_SIZE,
    dynamic  = False,   # fixed input shape for deterministic edge inference
    simplify = True,    # onnx-simplifier reduces graph complexity
    opset    = 17,
)
print('ONNX model exported →', onnx_path)

In [ ]:
import shutil

# Copy ONNX model and labels to edge/linux/models/
dest_model  = MODELS_DIR / 'trash_detector.onnx'
dest_labels = MODELS_DIR / 'labels.txt'

shutil.copy(onnx_path, dest_model)
dest_labels.write_text('\n'.join(CLASSES) + '\n')

print('Copied to', dest_model)
print('Labels  →', dest_labels)

## 6. Verify ONNX inference with onnxruntime

In [ ]:
import cv2
import numpy as np
import onnxruntime as ort

sess = ort.InferenceSession(
    str(dest_model),
    providers=['CUDAExecutionProvider', 'CPUExecutionProvider'],
)

input_name  = sess.get_inputs()[0].name
output_name = sess.get_outputs()[0].name
print('Input  :', input_name, sess.get_inputs()[0].shape)
print('Output :', output_name, sess.get_outputs()[0].shape)

# Dummy inference with a random frame
dummy = np.random.rand(1, 3, IMG_SIZE, IMG_SIZE).astype(np.float32)
out   = sess.run([output_name], {input_name: dummy})[0]
print('Output shape:', out.shape)  # expect [1, 84, 8400]

# Quick sanity: highest confidence across all anchors
pred         = out[0].T                            # [8400, 84]
scores       = pred[:, 4:].max(axis=1)
top_idx      = scores.argmax()
top_class    = pred[top_idx, 4:].argmax()
print(f'Highest confidence: {scores[top_idx]:.4f} → class {top_class} ({CLASSES[top_class]})')

## 7. Visualise predictions on a sample image

In [ ]:
import glob

# Grab the first validation image
val_images = glob.glob(str(ROOT / 'valid' / 'images' / '*.*'))
if not val_images:
    val_images = glob.glob(str(ROOT / 'train' / 'images' / '*.*'))

sample_path = val_images[0]
frame       = cv2.imread(sample_path)
h_orig, w_orig = frame.shape[:2]

# Preprocess — letterbox
s     = IMG_SIZE
scale = min(s / w_orig, s / h_orig)
nw, nh = int(w_orig * scale), int(h_orig * scale)
pad_x, pad_y = (s - nw) // 2, (s - nh) // 2
canvas = np.full((s, s, 3), 114, dtype=np.uint8)
canvas[pad_y:pad_y+nh, pad_x:pad_x+nw] = cv2.resize(frame, (nw, nh))
blob   = (canvas[:, :, ::-1].astype(np.float32) / 255.0)
blob   = np.transpose(blob, (2, 0, 1))[np.newaxis]

# Inference
raw   = sess.run([output_name], {input_name: blob})[0][0].T  # [8400, 84]
confs = raw[:, 4:].max(axis=1)
clsids = raw[:, 4:].argmax(axis=1)
mask  = confs >= 0.45

CATEGORY_COLOR = {
    'recycle':   (76, 175, 80),
    'compost':   (141, 110, 99),
    'landfill':  (158, 158, 158),
    'hazardous': (244, 67, 54),
}

vis = frame.copy()
for i in np.where(mask)[0]:
    cx, cy, bw, bh = raw[i, :4]
    x1 = int(((cx - bw/2) - pad_x) / scale)
    y1 = int(((cy - bh/2) - pad_y) / scale)
    x2 = int(((cx + bw/2) - pad_x) / scale)
    y2 = int(((cy + bh/2) - pad_y) / scale)
    label    = CLASSES[clsids[i]]
    category = LABEL_TO_CATEGORY[label]
    color    = CATEGORY_COLOR[category]
    cv2.rectangle(vis, (x1, y1), (x2, y2), color, 2)
    cv2.putText(vis, f'{label} {confs[i]:.2f}', (x1, y1-5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

plt.figure(figsize=(10, 7))
plt.imshow(vis[:, :, ::-1])
plt.axis('off')
plt.title(f'Sample detections — {Path(sample_path).name}')
plt.tight_layout()
plt.show()

## 8. Deployment checklist

| Step | Command |
|------|---------|
| Copy model to device | `scp edge/linux/models/trash_detector.onnx user@arduinounoq:~/carbinwatcher/models/` |
| Flash MCU firmware | `cd edge/mcu && pio run -t upload` |
| Set env vars | `export BIN_LEFT=recycle BIN_RIGHT=landfill GEMINI_API_KEY=... S3_BUCKET=...` |
| Run detector | `cd edge/linux && python detect.py` |
| Monitor serial | `cd edge/mcu && pio device monitor` |

**Calibrate focal length** once per camera:  
Hold a 100 mm-wide object at a known distance (e.g. 300 mm).  
Measure `object_width_px` in the frame, then:  
`FOCAL_LENGTH_PX = object_width_px * 300 / 100`  
Set via `export FOCAL_LENGTH_PX=<value>`.